# Step 1.2 — Radar Parsing (Upgraded) ✅

| | |
|---|---|
| **Input** | `output/step_0/samples_index.json`, `sensor_meta.json` per sample |
| **Outputs** | `output/step_1/radar/<sample>/<channel>.json` — per-point radar data + calibration |
| | `output/step_1/radar_parsed_data.csv` — combined table (replaces Excel) |
| | `output/step_1/radar_channel_report.json` — completeness check |
| **Used by** | Step 2.2 (radar processing/filtering), Step 3.2 (radar tracking), Step 4.4 (fusion) |

---

### Two bugs fixed from the original version

1. **`dyn_prop` was reading the wrong array index.** nuScenes radar points have 18 fields in this order:
   ```
   0:x  1:y  2:z  3:dyn_prop  4:id  5:rcs  6:vx  7:vy  8:vx_comp  9:vy_comp
   10:is_quality_valid  11:ambig_state  12:x_rms  13:y_rms  14:invalid_state  15:pdh0  16:vx_rms  17:vy_rms
   ```
   The original code read `pt[10]` (is_quality_valid) and called it `dyn_prop`. Fixed to `pt[3]`.

2. **Velocity fields were compensated, not raw.** The original code used `pt[8], pt[9]` (`vx_comp`, `vy_comp` — ego-motion compensated) as `vx, vy`. For **TTC / closing-speed**, you need the **raw** Doppler velocity at `pt[6], pt[7]`, which is the true relative velocity to your vehicle. This version stores **both**, clearly labeled, so downstream TTC logic uses the correct one.

### What's new

- Each point now carries its channel's `sensor_to_ego_translation/rotation` and `ego_pose` — this calibration is stored here for a *downstream* notebook to consume (Step 2.2, Step 3.2, or a projection step) when it needs to transform radar points into ego/global frame or project them onto camera images. This notebook does not call `project_point_to_camera()` itself; nothing here performs a projection.
- Derived fields (`range`, `azimuth`, `relative_speed`) are computed once in the main loop, so every output file has them — not only the Excel export.
- A diagnostic cell shows how many points nuScenes' default filtering silently drops, for reproducibility in your write-up.

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Verify config.py exists, load the master sample index
# ─────────────────────────────────────────────────────────────────

import json
from pathlib import Path

if not Path("config.py").exists():
    raise FileNotFoundError(
        "config.py not found in this folder. Copy it from the repo root "
        "and set DATA_ROOT to your nuScenes dataset path."
    )

from config import DATA_ROOT, NUSCENES_VERSION, STEP0_DIR, STEP1_DIR

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

RADAR_OUT_DIR = STEP1_DIR / "radar"
RADAR_OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(STEP0_DIR / "samples_index.json") as f:
    samples_index = json.load(f)

print(f"✅ config.py found")
print(f"✅ DATA_ROOT     : {DATA_ROOT}")
print(f"✅ STEP0_DIR     : {STEP0_DIR}")
print(f"✅ RADAR_OUT_DIR : {RADAR_OUT_DIR}")
print(f"✅ Loaded {len(samples_index)} samples from index.")


config.py loaded. PROJECT_ROOT = F:\Sensor fusion Research
DATA_ROOT   = F:\Sensor fusion Research\DATA SET\archive
OUTPUT_ROOT = F:\Sensor fusion Research\output


✅ config.py found
✅ DATA_ROOT     : F:\Sensor fusion Research\DATA SET\archive
✅ STEP0_DIR     : F:\Sensor fusion Research\output\step_0
✅ RADAR_OUT_DIR : F:\Sensor fusion Research\output\step_1\radar
✅ Loaded 404 samples from index.


In [2]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Diagnostic: how many points does nuScenes' default filter drop?
# RadarPointCloud.from_file() silently applies default filtering
# (invalid_state=[0], dynprop_states=range(7), ambig_states=[3]) before
# you ever see the array. This makes that visible for one sample.
# Purely informational — Cell 3 does not depend on anything defined here,
# so this cell can be skipped or deleted without affecting the pipeline.
# ─────────────────────────────────────────────────────────────────

from nuscenes.utils.data_classes import RadarPointCloud

sample_id = list(samples_index.keys())[0]
sensor_meta_path = STEP0_DIR / samples_index[sample_id]["folder"] / "sensor_meta.json"
with open(sensor_meta_path) as f:
    sensor_meta = json.load(f)

radar_meta = sensor_meta["sensor_channels"]["RADAR_FRONT"]
radar_path = DATA_ROOT / radar_meta["filename"]

# Default (filtered) load — what your pipeline actually uses
pc_filtered = RadarPointCloud.from_file(str(radar_path))

# Unfiltered load — disable all default quality filters
pc_raw = RadarPointCloud.from_file(
    str(radar_path),
    invalid_states=list(range(18)),
    dynprop_states=list(range(8)),
    ambig_states=list(range(5))
)

n_filtered = pc_filtered.points.shape[1]
n_raw = pc_raw.points.shape[1]

print(f"RADAR_FRONT, {sample_id}:")
print(f"  Raw points in file        : {n_raw}")
print(f"  After default quality filter: {n_filtered}")
print(f"  Dropped by devkit defaults : {n_raw - n_filtered} "
      f"({(n_raw - n_filtered) / max(n_raw,1) * 100:.1f}%)")
print("\n👉 This filtering happens BEFORE your own 'valid' check — worth")
print("   noting in your methodology write-up for reproducibility.")


RADAR_FRONT, sample_0000:
  Raw points in file        : 125
  After default quality filter: 74
  Dropped by devkit defaults : 51 (40.8%)

👉 This filtering happens BEFORE your own 'valid' check — worth
   noting in your methodology write-up for reproducibility.


In [3]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Main pipeline: parse radar points with FIXED indices,
# both raw and compensated velocity, and calibration attached.
# ─────────────────────────────────────────────────────────────────

import math
import numpy as np
from nuscenes.utils.data_classes import RadarPointCloud

total_points_written = 0
total_channels_written = 0
failed_channels = []

for sample_id, info in samples_index.items():
    # FIXED — join with STEP0_DIR since "folder" is a relative name
    sensor_meta_path = STEP0_DIR / info["folder"] / "sensor_meta.json"
    with open(sensor_meta_path) as f:
        sensor_meta = json.load(f)

    radar_channels = [s for s in info["sensors"] if s.startswith("RADAR_")]
    if not radar_channels:
        continue

    radar_out_sample = RADAR_OUT_DIR / sample_id
    radar_out_sample.mkdir(parents=True, exist_ok=True)

    for radar_channel in radar_channels:
        meta = sensor_meta["sensor_channels"][radar_channel]
        radar_data_path = DATA_ROOT / meta["filename"]

        try:
            radar_pc = RadarPointCloud.from_file(str(radar_data_path))
            radar_array = radar_pc.points.T   # shape (N, 18)

            # Calibration — needed for ego/global transform and camera projection
            calibration = {
                "sensor_to_ego_translation": meta["sensor_to_ego_translation"],
                "sensor_to_ego_rotation": meta["sensor_to_ego_rotation"],
                "ego_pose": meta["ego_pose"]
            }

            parsed_points = []
            for pt in radar_array:
                x, y, z = float(pt[0]), float(pt[1]), float(pt[2])
                dyn_prop = int(pt[3])                       # FIXED: was pt[10]
                vx_raw, vy_raw = float(pt[6]), float(pt[7])  # raw Doppler velocity — use for TTC
                vx_comp, vy_comp = float(pt[8]), float(pt[9])  # ego-compensated — use for moving/stationary classification
                is_quality_valid = int(pt[10])               # FIXED: correctly labeled now

                valid = (not math.isnan(vx_raw)) and (not math.isnan(vy_raw))

                range_val = math.hypot(x, y)
                azimuth_deg = math.degrees(math.atan2(y, x))
                # Closing speed relative to ego — uses RAW velocity (correct for TTC)
                relative_speed = (x * vx_raw + y * vy_raw) / range_val if range_val > 0 else 0.0

                parsed_points.append({
                    "x": x, "y": y, "z": z,
                    "vx": vx_raw, "vy": vy_raw,                  # relative velocity (for TTC)
                    "vx_comp": vx_comp, "vy_comp": vy_comp,      # compensated velocity (for motion classification)
                    "dyn_prop": dyn_prop,
                    "is_quality_valid": is_quality_valid,
                    "valid": valid,
                    "range": round(range_val, 3),
                    "azimuth_deg": round(azimuth_deg, 2),
                    "relative_speed": round(relative_speed, 3),
                })

            out_data = {
                "sample_id": sample_id,
                "radar_channel": radar_channel,
                "calibration": calibration,
                "points": parsed_points
            }

            out_file = radar_out_sample / f"{radar_channel}.json"
            with open(out_file, "w") as f:
                json.dump(out_data, f, indent=2)

            total_points_written += len(parsed_points)
            total_channels_written += 1

        except Exception as e:
            print(f"❌ Failed parsing {radar_channel} for {sample_id}: {e}")
            failed_channels.append({"sample_id": sample_id, "channel": radar_channel, "error": str(e)})
            continue

assert total_channels_written > 0, "No radar channels were written — check DATA_ROOT and file paths"

print(f"✅ Step 1.2 complete.")
print(f"   Channels written : {total_channels_written}")
print(f"   Total points     : {total_points_written}")

if failed_channels:
    with open(RADAR_OUT_DIR / "parse_failures.json", "w") as f:
        json.dump(failed_channels, f, indent=2)
    print(f"⚠️ {len(failed_channels)} channel-parses failed — see parse_failures.json")
else:
    with open(RADAR_OUT_DIR / "parse_failures.json", "w") as f:
        json.dump(failed_channels, f, indent=2)
    print(f"✅ No parse failures — parse_failures.json written empty.")


✅ Step 1.2 complete.
   Channels written : 2020
   Total points     : 79519
✅ No parse failures — parse_failures.json written empty.


In [4]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — Combined CSV export (replaces Excel — faster, smaller, git-friendly)
# Excel kept as optional for manual inspection only.
# ─────────────────────────────────────────────────────────────────

import pandas as pd

all_rows = []
for sample_folder in sorted(RADAR_OUT_DIR.iterdir()):
    if not sample_folder.is_dir():
        continue
    for radar_file in sample_folder.glob("*.json"):
        with open(radar_file) as f:
            data = json.load(f)
        for pt in data["points"]:
            row = dict(pt)
            row["sample_id"] = data["sample_id"]
            row["radar_channel"] = data["radar_channel"]
            all_rows.append(row)

radar_df = pd.DataFrame(all_rows)

# Save full CSV (fast, portable)
csv_path = STEP1_DIR / "radar_parsed_data.csv"
radar_df.to_csv(csv_path, index=False)
print(f"✅ Combined CSV saved: {csv_path}")
print(f"   {len(radar_df)} total points across {radar_df['radar_channel'].nunique()} channel types")

# Optional filtered view — valid + moving points only (uses raw velocity)
filtered_df = radar_df[radar_df["valid"] & ((radar_df["vx"] != 0) | (radar_df["vy"] != 0))]
print(f"   {len(filtered_df)} valid + moving points ({len(filtered_df)/len(radar_df)*100:.1f}%)")

display(radar_df.head())

✅ Combined CSV saved: F:\Sensor fusion Research\output\step_1\radar_parsed_data.csv
   79519 total points across 5 channel types
   67008 valid + moving points (84.3%)


,x,y,z,vx,vy,vx_comp,vy_comp,dyn_prop,is_quality_valid,valid,range,azimuth_deg,relative_speed,sample_id,radar_channel
0,8.6,-2.9,0.0,8.75,-0.25,0.243894,-0.082243,1,1,True,9.076,-18.63,8.371,sample_0000,RADAR_BACK_LEFT
1,9.0,7.3,0.0,9.75,-0.25,-0.004036,-0.003273,1,1,True,11.588,39.05,7.415,sample_0000,RADAR_BACK_LEFT
2,12.6,-0.1,0.0,8.75,-0.25,-0.094915,0.000753,1,1,True,12.600,-0.45,8.752,sample_0000,RADAR_BACK_LEFT
3,14.0,0.7,0.0,9.00,-0.25,0.090357,0.004518,1,1,True,14.017,2.86,8.976,sample_0000,RADAR_BACK_LEFT
4,14.8,7.9,0.0,9.50,-0.25,0.040494,0.021615,1,1,True,16.776,28.09,8.263,sample_0000,RADAR_BACK_LEFT


In [5]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Check all 5 radar channels are present per sample
# ─────────────────────────────────────────────────────────────────

expected_radar_channels = {
    "RADAR_FRONT", "RADAR_FRONT_LEFT", "RADAR_FRONT_RIGHT",
    "RADAR_BACK_LEFT", "RADAR_BACK_RIGHT"
}

missing_report = {}
for sample_folder in sorted(RADAR_OUT_DIR.iterdir()):
    if not sample_folder.is_dir():
        continue
    present = {f.stem for f in sample_folder.glob("*.json")}
    missing = expected_radar_channels - present
    if missing:
        missing_report[sample_folder.name] = sorted(missing)

if not missing_report:
    print("✅ All samples have all 5 radar channels.")
else:
    print(f"⚠️ {len(missing_report)} samples missing radar channels:")
    for sid, missing in sorted(missing_report.items())[:10]:
        print(f"  - {sid}: missing {missing}")

report_path = STEP1_DIR / "radar_channel_report.json"
with open(report_path, "w") as f:
    json.dump(missing_report, f, indent=2)
print(f"📝 Report saved to: {report_path}")

✅ All samples have all 5 radar channels.
📝 Report saved to: F:\Sensor fusion Research\output\step_1\radar_channel_report.json
